In [1]:
# Enable auto-reload for development
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys
import os

# Add scripts directory to path
sys.path.append(os.path.abspath("../../scripts"))

# Import modules
from data_pipeline import load_and_prepare_data, load_single_channel
import xgb_scripts
import config

In [2]:
unique_frequencies = [
    2.54e-01, 3.40e-01, 4.56e-01, 6.12e-01, 8.22e-01, 9.99e-01, 1.10e+00, 1.33e+00,
    1.48e+00, 1.78e+00, 1.99e+00, 2.37e+00, 2.66e+00, 3.16e+00, 3.57e+00, 4.22e+00,
    4.80e+00, 5.62e+00, 6.43e+00, 7.50e+00, 8.64e+00, 1.00e+01, 1.16e+01, 1.33e+01,
    1.55e+01, 1.78e+01, 2.09e+01, 2.37e+01, 2.80e+01, 3.16e+01, 3.75e+01, 4.22e+01,
    5.03e+01, 5.62e+01, 6.76e+01, 7.50e+01, 9.06e+01, 1.02e+02, 1.22e+02, 1.35e+02,
    1.63e+02, 1.78e+02, 2.19e+02, 2.37e+02, 2.94e+02, 3.16e+02, 3.94e+02, 4.22e+02,
    5.29e+02, 5.64e+02, 7.10e+02, 7.50e+02, 9.52e+02, 1.00e+03, 1.28e+03, 1.33e+03,
    1.71e+03, 1.78e+03, 2.30e+03, 2.37e+03, 3.09e+03, 3.16e+03, 4.14e+03, 4.22e+03,
    5.56e+03, 5.62e+03, 7.45e+03, 7.50e+03, 1.00e+04
]

In [12]:
df = load_single_channel(data_folder="../../data/04-03-24", channel="A1")
df = (
  df.dropna()
    .query("`freq/Hz` > 0.2 and `freq/Hz` <= 20000")
    .query("Ns in [6]")
)

df[df['cycle number']==42][['freq/Hz','time/s']]


,freq/Hz,time/s
12631,10000.000,373000.0
12632,7500.000,373000.0
12633,5620.000,373000.0
12634,4220.000,373000.0
12635,3160.000,373000.0
12636,2370.000,373000.0
12637,1780.000,373000.0
12638,1330.000,373000.0
12639,1000.000,373000.0
12640,750.000,373000.0


In [15]:
df = load_single_channel(data_folder="../../data/04-03-24", channel="A1")

valid_cycles = sorted(df.loc[df['cycle number'] >= 1, 'cycle number'].unique())

for cycle in valid_cycles:
    sub = df[(df['cycle number'] == cycle) & (df['Ns'] == 1)]
    vc = sub.groupby('freq/Hz').size().value_counts()
    print(f"cycle {cycle}:")
    print(vc, "\n")


cycle 1.0:
1    48
Name: count, dtype: int64 

cycle 2.0:
1    48
Name: count, dtype: int64 

cycle 3.0:
1    48
Name: count, dtype: int64 

cycle 4.0:
1    48
Name: count, dtype: int64 

cycle 5.0:
1    48
Name: count, dtype: int64 

cycle 6.0:
1    48
Name: count, dtype: int64 

cycle 7.0:
1    48
Name: count, dtype: int64 

cycle 8.0:
1    48
Name: count, dtype: int64 

cycle 9.0:
1    48
Name: count, dtype: int64 

cycle 10.0:
1    48
Name: count, dtype: int64 

cycle 11.0:
1    48
Name: count, dtype: int64 

cycle 12.0:
1    48
Name: count, dtype: int64 

cycle 13.0:
1    48
Name: count, dtype: int64 

cycle 14.0:
1    48
Name: count, dtype: int64 

cycle 15.0:
1    48
Name: count, dtype: int64 

cycle 16.0:
1    48
Name: count, dtype: int64 

cycle 17.0:
1    48
Name: count, dtype: int64 

cycle 18.0:
1    48
Name: count, dtype: int64 

cycle 19.0:
1    48
Name: count, dtype: int64 

cycle 20.0:
1    48
Name: count, dtype: int64 

cycle 21.0:
1    48
Name: count, dtype: int64 

c

In [14]:
df = load_single_channel(data_folder="../../data/04-03-24", channel="A8")


df_ns1 = (
    df.dropna()
        .query("`freq/Hz` > 0.2 and `freq/Hz` <= 20000")
        .query("Ns in [1]")
)

dup_counts = (
    df_ns1.groupby(["cycle number", "freq/Hz"])
        .size()
        .reset_index(name="count")
)

dup_counts[dup_counts["count"] > 1]


,cycle number,freq/Hz,count


In [18]:
cycle_dupes = dup_counts[dup_counts["count"] > 1]["cycle number"].value_counts()
print("Number of cycles with duplicates:", len(cycle_dupes))
print(cycle_dupes.head())

Number of cycles with duplicates: 0
Series([], Name: count, dtype: int64)


## Exploring relevant frequencies

## Systematic Frequency Selection Methodology

Following research best practices to determine optimal frequencies for our specific dataset.
This approach combines multiple statistical and physical criteria rather than just copying literature values.

## Comprehensive Feature Importance Analysis Using All Cycles Model

Loading the XGBoost model trained on ALL CYCLES (1-267) with binning + all frequencies to understand which specific frequencies and components are most important across the complete battery degradation spectrum - from early life through end-of-life conditions.

In [3]:
import joblib

models = joblib.load("../../models/xgb_temporal.pkl")
print(f"\nSuccessfully loaded XGBoost ensemble with {len(models)} models")
print(f"Model type: {type(models[0])}")


Successfully loaded XGBoost ensemble with 10 models
Model type: <class 'xgboost.sklearn.XGBRegressor'>


In [5]:
# Load data with ALL CYCLES to match the saved model configuration
X_train, X_test, y_train, y_test = load_and_prepare_data(
    data_folder="../../data/04-03-24", 
    method="temporal",
)

X_train: (1754, 74), y_train: (1754,)
X_test: (444, 74), y_test: (444,)
Train capacity range: 114.0 - 4050.0 mAh
Test capacity range: 78.0 - 3570.0 mAh


In [6]:
# Get feature importance from the loaded models
feature_importance = np.mean([model.feature_importances_ for model in models], axis=0)
print(f"Feature importance extracted from {len(models)} models")

# Sort features by importance
feature_indices = np.argsort(feature_importance)[::-1]  # Descending order
print(f"\nTop 10 most important features:")
for i in range(10):
    idx = feature_indices[i]
    print(f"{i+1:2d}. Feature {idx:3d}: Importance = {feature_importance[idx]:.4f}")

# Show cumulative importance
cumulative_importance = np.cumsum(feature_importance[feature_indices])
total_importance = np.sum(feature_importance)

print(f"\nCumulative importance breakdown:")
for n_features in [5, 10, 15, 20, 30]:
    if n_features <= len(feature_importance):
        pct = cumulative_importance[n_features-1] / total_importance * 100
        print(f"Top {n_features:2d} features capture {pct:.1f}% of total importance")

Feature importance extracted from 10 models

Top 10 most important features:
 1. Feature  35: Importance = 0.5677
 2. Feature  33: Importance = 0.1430
 3. Feature  32: Importance = 0.1235
 4. Feature  31: Importance = 0.0926
 5. Feature  30: Importance = 0.0339
 6. Feature  34: Importance = 0.0139
 7. Feature  28: Importance = 0.0108
 8. Feature  72: Importance = 0.0042
 9. Feature   0: Importance = 0.0036
10. Feature  36: Importance = 0.0012

Cumulative importance breakdown:
Top  5 features capture 96.1% of total importance
Top 10 features capture 99.5% of total importance
Top 15 features capture 99.7% of total importance
Top 20 features capture 99.8% of total importance
Top 30 features capture 99.9% of total importance


In [ ]:
feature_details = []
for i in range(len(feature_importance)):
    if i < 69:  # Real impedance
        freq = unique_frequencies[i]
        feature_type = "Real"
    elif i < 138:  # Imaginary impedance  
        freq = unique_frequencies[i - 69]
        feature_type = "Imaginary"
    else:  # Action vector
        freq = None
        feature_type = "Action"
    
    feature_details.append({
        'idx': i,
        'type': feature_type,
        'frequency': freq,
        'importance': feature_importance[i]
    })

# Sort by importance (descending)
feature_details.sort(key=lambda x: x['importance'], reverse=True)

print("ALL FEATURES RANKED BY IMPORTANCE:")
print("Rank | Feature | Type      | Frequency (Hz) | Importance | % of Total")

total_imp = sum(feature_importance)
cumulative = 0

for rank, feature in enumerate(feature_details, 1):
    cumulative += feature['importance']
    pct_individual = (feature['importance'] / total_imp) * 100
    pct_cumulative = (cumulative / total_imp) * 100
    
    if feature['frequency'] is not None:
        freq_str = f"{feature['frequency']:>8.2f}"
    else:
        freq_str = "   Action"
    
    print(f"{rank:4d} | {feature['idx']:7d} | {feature['type']:<9} | {freq_str} | {feature['importance']:10.4f} | {pct_individual:5.1f}% ({pct_cumulative:5.1f}%)")


ALL FEATURES RANKED BY IMPORTANCE:
Rank | Feature | Type      | Frequency (Hz) | Importance | % of Total
   1 |      81 | Imaginary |     2.66 |     0.4181 |  41.8% ( 41.8%)
   2 |      48 | Real      |   529.00 |     0.3639 |  36.4% ( 78.2%)
   3 |      90 | Imaginary |    10.00 |     0.0402 |   4.0% ( 82.2%)
   4 |      18 | Real      |     6.43 |     0.0327 |   3.3% ( 85.5%)
   5 |      68 | Real      | 10000.00 |     0.0297 |   3.0% ( 88.5%)
   6 |      66 | Real      |  7450.00 |     0.0149 |   1.5% ( 89.9%)
   7 |       4 | Real      |     0.82 |     0.0116 |   1.2% ( 91.1%)
   8 |      64 | Real      |  5560.00 |     0.0100 |   1.0% ( 92.1%)
   9 |      15 | Real      |     4.22 |     0.0083 |   0.8% ( 92.9%)
  10 |      30 | Real      |    37.50 |     0.0072 |   0.7% ( 93.7%)
  11 |      67 | Real      |  7500.00 |     0.0059 |   0.6% ( 94.2%)
  12 |     132 | Imaginary |  4220.00 |     0.0053 |   0.5% ( 94.8%)
  13 |      20 | Real      |     8.64 |     0.0035 |   0.4% ( 95.1%

In [6]:
import json


# Prepare data
feature_data = []
cumulative = 0
total_imp = sum(feature_importance)

for rank, feature in enumerate(feature_details, 1):
    cumulative += feature['importance']
    pct_individual = (feature['importance'] / total_imp) * 100
    pct_cumulative = (cumulative / total_imp) * 100
    
    feature_data.append({
        'rank': rank,
        'feature_index': int(feature['idx']),
        'feature_type': feature['type'],
        'frequency_hz': float(feature['frequency']) if feature['frequency'] is not None else 'Action_Vector',
        'importance': float(feature['importance']),
        'importance_percent': float(pct_individual),
        'cumulative_percent': float(pct_cumulative)
    })

# Save CSV
df_features = pd.DataFrame(feature_data)
df_features.to_csv("../../results/feature_relevance.csv", index=False)

# Save JSON
with open("../../results/feature_relevance.json", 'w') as f:
    json.dump(feature_data, f, indent=2)

print("Data exported to feature_relevance.csv and feature_relevance.json")

Data exported to feature_relevance.csv and feature_relevance.json


In [11]:
models = joblib.load("../../models/xgb_binning.pkl")
print(f"\nSuccessfully loaded XGBoost ensemble with {len(models)} models")
print(f"Model type: {type(models[0])}")



Successfully loaded XGBoost ensemble with 10 models
Model type: <class 'xgboost.sklearn.XGBRegressor'>


In [9]:

# Load data with ALL CYCLES to match the saved model configuration
X_train, X_test, y_train, y_test = load_and_prepare_data(
    data_folder="../../data/04-03-24", 
    method="bin_and_split",
)


Successfully loaded XGBoost ensemble with 10 models
Model type: <class 'xgboost.sklearn.XGBRegressor'>
X_train: (1727, 74), y_train: (1727,)
X_test: (471, 74), y_test: (471,)
Train capacity range: 80.7 - 4050.0 mAh
Test capacity range: 78.0 - 3850.0 mAh


In [12]:
# Get feature importance from the loaded models
feature_importance = np.mean([model.feature_importances_ for model in models], axis=0)
print(f"Feature importance extracted from {len(models)} models")

# Sort features by importance
feature_indices = np.argsort(feature_importance)[::-1]  # Descending order
print(f"\nTop 10 most important features:")
for i in range(10):
    idx = feature_indices[i]
    print(f"{i+1:2d}. Feature {idx:3d}: Importance = {feature_importance[idx]:.4f}")

# Show cumulative importance
cumulative_importance = np.cumsum(feature_importance[feature_indices])
total_importance = np.sum(feature_importance)

print(f"\nCumulative importance breakdown:")
for n_features in [5, 10, 15, 20, 30]:
    if n_features <= len(feature_importance):
        pct = cumulative_importance[n_features-1] / total_importance * 100
        print(f"Top {n_features:2d} features capture {pct:.1f}% of total importance")

Feature importance extracted from 10 models

Top 10 most important features:
 1. Feature  32: Importance = 0.4759
 2. Feature  33: Importance = 0.2131
 3. Feature  35: Importance = 0.1084
 4. Feature  30: Importance = 0.0899
 5. Feature  34: Importance = 0.0593
 6. Feature  31: Importance = 0.0123
 7. Feature  29: Importance = 0.0083
 8. Feature  36: Importance = 0.0068
 9. Feature   5: Importance = 0.0062
10. Feature   1: Importance = 0.0029

Cumulative importance breakdown:
Top  5 features capture 94.6% of total importance
Top 10 features capture 98.3% of total importance
Top 15 features capture 99.2% of total importance
Top 20 features capture 99.5% of total importance
Top 30 features capture 99.8% of total importance


In [ ]:
df = load_single_channel(data_folder="../../data/04-03-24", channel="A8")


df_ns1 = (
  df.dropna()
    .query("`freq/Hz` > 0.2 and `freq/Hz` <= 20000")
    .query("Ns in [1]")
)

df_ns6 = (
  df.dropna()
    .query("`freq/Hz` > 0.2 and `freq/Hz` <= 20000")
    .query("Ns in [6]")
)

unique_freqs_ns1 = np.array(df_ns1['freq/Hz'].unique())
unique_freqs_ns1.sort()


unique_freqs_ns6 = np.array(df_ns6['freq/Hz'].unique())
unique_freqs_ns6.sort()

print("Unique frequencies for Ns=1:")
print(unique_freqs_ns1)
print("\n")
print(len(unique_freqs_ns1))
print("\n")
print("Unique frequencies for Ns=6:")
print(unique_freqs_ns6)
print("\n")
print(len(unique_freqs_ns6))



Unique frequencies for Ns=1:
[2.54e-01 3.40e-01 4.56e-01 6.12e-01 8.22e-01 1.10e+00 1.48e+00 1.99e+00
 2.66e+00 3.57e+00 4.80e+00 6.43e+00 8.64e+00 1.16e+01 1.55e+01 2.09e+01
 2.80e+01 3.75e+01 5.03e+01 6.76e+01 9.06e+01 1.22e+02 1.63e+02 2.19e+02
 2.94e+02 3.94e+02 5.29e+02 7.10e+02 9.52e+02 1.28e+03 1.71e+03 2.30e+03
 3.09e+03 4.14e+03 5.56e+03 7.45e+03 1.00e+04]


37


Unique frequencies for Ns=6:
[9.99e-01 1.33e+00 1.78e+00 2.37e+00 3.16e+00 4.22e+00 5.62e+00 7.50e+00
 1.00e+01 1.33e+01 1.78e+01 2.37e+01 3.16e+01 4.22e+01 5.62e+01 7.50e+01
 1.02e+02 1.35e+02 1.78e+02 2.37e+02 3.16e+02 4.22e+02 5.64e+02 7.50e+02
 1.00e+03 1.33e+03 1.78e+03 2.37e+03 3.16e+03 4.22e+03 5.62e+03 7.50e+03
 1.00e+04]


33


In [28]:
unique_diff = np.setdiff1d(unique_freqs_ns1, unique_freqs_ns6)
print("Frequencies in Ns=1 not in Ns=6:")
print(unique_diff)
print("\n")
unique_diff = np.setdiff1d(unique_freqs_ns6, unique_freqs_ns1)
print("Frequencies in Ns=6 not in Ns=1:")
print(unique_diff)

Frequencies in Ns=1 not in Ns=6:
[2.54e-01 3.40e-01 4.56e-01 6.12e-01 8.22e-01 1.10e+00 1.48e+00 1.99e+00
 2.66e+00 3.57e+00 4.80e+00 6.43e+00 8.64e+00 1.16e+01 1.55e+01 2.09e+01
 2.80e+01 3.75e+01 5.03e+01 6.76e+01 9.06e+01 1.22e+02 1.63e+02 2.19e+02
 2.94e+02 3.94e+02 5.29e+02 7.10e+02 9.52e+02 1.28e+03 1.71e+03 2.30e+03
 3.09e+03 4.14e+03 5.56e+03 7.45e+03]


Frequencies in Ns=6 not in Ns=1:
[9.99e-01 1.33e+00 1.78e+00 2.37e+00 3.16e+00 4.22e+00 5.62e+00 7.50e+00
 1.00e+01 1.33e+01 1.78e+01 2.37e+01 3.16e+01 4.22e+01 5.62e+01 7.50e+01
 1.02e+02 1.35e+02 1.78e+02 2.37e+02 3.16e+02 4.22e+02 5.64e+02 7.50e+02
 1.00e+03 1.33e+03 1.78e+03 2.37e+03 3.16e+03 4.22e+03 5.62e+03 7.50e+03]


## Most Important Frequencies for Ns=1 Models

Mapping feature indices to actual frequencies to understand which frequencies drive model predictions.

In [29]:
# Load temporal model feature importance
models_temporal = joblib.load("../../models/xgb_temporal.pkl")
fi_temporal = np.mean([model.feature_importances_ for model in models_temporal], axis=0)

# Load binning model feature importance
models_binning = joblib.load("../../models/xgb_binning.pkl")
fi_binning = np.mean([model.feature_importances_ for model in models_binning], axis=0)

# Map top features to frequencies
top_n = 10
print("TOP 10 MOST IMPORTANT FEATURES\n")

print("TEMPORAL MODEL:")
for rank, idx in enumerate(np.argsort(fi_temporal)[::-1][:top_n], 1):
    if idx < 37:
        freq = unique_freqs_ns1[idx]
        feature_type = "Real"
    elif idx < 74:
        freq = unique_freqs_ns1[idx - 37]
        feature_type = "Imaginary"
    print(f"{rank:2d}. Feature {idx:3d} | {feature_type:9s} | {freq:8.2f} Hz | Importance: {fi_temporal[idx]:.4f}")

print("\nBINNING MODEL:")
for rank, idx in enumerate(np.argsort(fi_binning)[::-1][:top_n], 1):
    if idx < 37:
        freq = unique_freqs_ns1[idx]
        feature_type = "Real"
    elif idx < 74:
        freq = unique_freqs_ns1[idx - 37]
        feature_type = "Imaginary"
    print(f"{rank:2d}. Feature {idx:3d} | {feature_type:9s} | {freq:8.2f} Hz | Importance: {fi_binning[idx]:.4f}")

TOP 10 MOST IMPORTANT FEATURES

TEMPORAL MODEL:
 1. Feature  35 | Real      |  7450.00 Hz | Importance: 0.5677
 2. Feature  33 | Real      |  4140.00 Hz | Importance: 0.1430
 3. Feature  32 | Real      |  3090.00 Hz | Importance: 0.1235
 4. Feature  31 | Real      |  2300.00 Hz | Importance: 0.0926
 5. Feature  30 | Real      |  1710.00 Hz | Importance: 0.0339
 6. Feature  34 | Real      |  5560.00 Hz | Importance: 0.0139
 7. Feature  28 | Real      |   952.00 Hz | Importance: 0.0108
 8. Feature  72 | Imaginary |  7450.00 Hz | Importance: 0.0042
 9. Feature   0 | Real      |     0.25 Hz | Importance: 0.0036
10. Feature  36 | Real      | 10000.00 Hz | Importance: 0.0012

BINNING MODEL:
 1. Feature  32 | Real      |  3090.00 Hz | Importance: 0.4759
 2. Feature  33 | Real      |  4140.00 Hz | Importance: 0.2131
 3. Feature  35 | Real      |  7450.00 Hz | Importance: 0.1084
 4. Feature  30 | Real      |  1710.00 Hz | Importance: 0.0899
 5. Feature  34 | Real      |  5560.00 Hz | Importance